In [15]:
import sys
sys.path.append('../../utils')
from functions import * 

In [16]:
from importlib import reload
import sys
import scipy as sp

# Path to the Leaflet repository
PATH_TO_LEAFLET_REPO = '/gpfs/commons/home/kisaev/Leaflet/src/beta-binomial-mix/'
sys.path.append(PATH_TO_LEAFLET_REPO)

In [17]:
import cell_state_asign_consistency
reload  (cell_state_asign_consistency)

import betabinomo_mix_singlecells
reload (betabinomo_mix_singlecells)

from importlib import reload
from load_cluster_data import load_cluster_data
from betabinomo_mix_singlecells import *
#reload(betabinomo_mix_singlecells)
from cell_state_asign_consistency import *
#reload(cell_state_asign_consistency)
import torch
import sklearn.manifold 
import plotnine as p9
import time
# indicate plot should be small 4 by 4
import plotnine as p9
from plotnine import ggplot, geom_point, aes, stat_smooth, facet_wrap, geom_violin, theme, element_blank, geom_text, geom_bar, geom_hline
import plotnine
from tqdm import tqdm
plotnine.options.figure_size = (4, 4)
import seaborn as sns
sns.set_theme(style="whitegrid")

### Some utility functions 


In [18]:
# write function that takes in Cluster name 
def check_SS_cluster(cluster_name):
    
    juncs_c = junc_info[junc_info["Cluster"] == cluster_name]
    
    # keep only rows where either start or end appear twice
    s = np.array(juncs_c[juncs_c.duplicated(subset=['start'])].start.unique())
    e = np.array(juncs_c[juncs_c.duplicated(subset=['end'])].end.unique())
    juncs_c = juncs_c[(juncs_c["start"].isin(s)) | (juncs_c["end"].isin(e))]

    # if num rows in juncs_c is 3 then return cluster name 
    if len(juncs_c) == 3:
        return cluster_name
    else:
        pass

In [ ]:
def simulate_junc_counts(cluster_counts, cell_types=None, simulate_cell_types=False, psi_prior_shape1=0.5, psi_prior_shape2=0.5):
    
    """Simulate junc counts while keeping the cluster counts of observed data. 
    
    Args: 
        cluster_counts: scipy coo_matrix. 
        cell_types: pandas Categorical series of pre-defined cell types to use for simulations 
        simulate_cell_types: bool. If True, simulate cell types with default K of 2.
    """
    
    N, P = cluster_counts.shape  # number of cells, number of junctions
     
    # use real cell types labels to represent intron clusters being higher/lower in specific cell types 
    if cell_types is not None:
        print("Using pre-defined cell types!")
        K = len(cell_types.cat.categories)  # number of cell types
    
    if simulate_cell_types:
        print("Simulating cell types!")
        K = 2  # going to simulate two cell types and randomly assign them to cells
        cell_types = pd.Categorical(np.random.choice(K, N))

    print(N, P, K, len(cell_types))
    
    # simulate PSI for each junction in each cell type via pre-defined beta distributions
    cell_type_psi = torch.distributions.beta.Beta(psi_prior_shape1, psi_prior_shape2).sample([P, K]) 
    print("Done simulating PSI!")

    # use real cluster counts to simulate junc counts with binomial distribution
    sim_junc_counts = cluster_counts.copy() 

    sim_junc_counts.data = torch.distributions.binomial.Binomial( 
         total_count=torch.tensor(cluster_counts.data), 
         probs=cell_type_psi[
             cluster_counts.col, 
             cell_types[cluster_counts.row] #use cell type of each cell to get the corresponding psi
         ]
    ).sample().numpy()
    
    print("Done simulating junc counts!")
    
    return sim_junc_counts, cell_types, cell_type_psi

### Settings and Load data

In [19]:
torch.manual_seed(42)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

K = 5 # set to very high number 

float_type = { 
    "device" : device, 
    "dtype" : torch.float, # save memory
}

hypers = {
    "eta" : 1./K, 
    "alpha_prior" : 1., # karin had 0.65 
    "pi_prior" : 1.
}

cpu


In [20]:
hypers["eta"]

0.2

### Load data

In [21]:
# this folder contains input data for each tissue cell type sample
input_files_folder = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/Brain/train/'

# read each file in the input_files_folder (h5) files and concatenate them all into the summarized_data object 
files = os.listdir(input_files_folder)
df_list = []
for file in files:
    if file.endswith(".h5"):
        path_with_quotes = input_files_folder + file
        fixed_path = path_with_quotes.replace("'", "")
        df = pd.read_hdf(fixed_path, 'df')
        df_list.append(df)
    else:
        pass
# concatenate all dataframes
summarized_data = pd.concat(df_list, ignore_index=True)
summarized_data.head()

,cell_id,Cluster,Cluster_Counts,junction_id,junc_count,cell_type,junc_ratio,cell_id_index,junction_id_index
0,A14-MAA000581-3_10_M-1-1_Brain_Non-Myeloid_bra...,422,16,1_34299838_34301860,8,Brain_Non-Myeloid_brain_pericyte,0.500000,92,35150
1,A14-MAA000581-3_10_M-1-1_Brain_Non-Myeloid_bra...,422,16,1_34301932_34302829,8,Brain_Non-Myeloid_brain_pericyte,0.500000,92,35152
2,A14-MAA000581-3_10_M-1-1_Brain_Non-Myeloid_bra...,423,30,1_34303032_34303420,16,Brain_Non-Myeloid_brain_pericyte,0.533333,92,35153
3,A14-MAA000581-3_10_M-1-1_Brain_Non-Myeloid_bra...,423,30,1_34303531_34306527,14,Brain_Non-Myeloid_brain_pericyte,0.466667,92,35155
4,A14-MAA000581-3_10_M-1-1_Brain_Non-Myeloid_bra...,469,28,1_36307836_36316208,7,Brain_Non-Myeloid_brain_pericyte,0.250000,92,35177


In [22]:
print(len(summarized_data.cell_id.unique())) # num cells 
print(len(summarized_data.junction_id.unique())) # num junctions
print(len(summarized_data.Cluster.unique())) # num Cluster

5499
78814
17976


In [23]:
final_data, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = load_cluster_data(
    input_folder = input_files_folder) 

Reading in data from folder ...
/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/Brain/train/


In [ ]:
# still do preprocessing in scipy
import scipy.sparse as sp
import matplotlib.pyplot as plt 

indices = (final_data.cell_id_index, final_data.junction_id_index)
indices_np = np.stack(indices)
junc_counts = sp.coo_matrix((final_data.junc_count, indices))
cluster_counts = sp.coo_matrix((final_data.cluster_count, indices))

### For simulating data, let's only keep clusters that have exon skipping event, so cluster with three junctions where 

In [ ]:
# SS are shared between end of J1 and start of J2 and end of J2 and start of J3
junc_info = summarized_data[["junction_id", "Cluster", "junction_id_index"]].drop_duplicates()

# get number of junctions in each cluster first 
cluster_junc_counts = junc_info.groupby(["Cluster"]).agg({"junction_id": "count"}).reset_index()
clusts_keep = cluster_junc_counts[cluster_junc_counts["junction_id"] == 3 ]
junc_info = junc_info[junc_info["Cluster"].isin(clusts_keep["Cluster"])]

# break up junction_id column in junc_info into chr, start and end 
junc_info["chr"] = junc_info["junction_id"].str.split("_").str[0]
junc_info["start"] = junc_info["junction_id"].str.split("_").str[1]
junc_info["end"] = junc_info["junction_id"].str.split("_").str[2]
print(len(junc_info["Cluster"].unique()))

In [ ]:
# run function on all clusters to find simple exon skipping events 
clusters_SS = []

for cluster in junc_info["Cluster"].unique():
    clusters_SS.append(check_SS_cluster(cluster))

# keep only entries in clusters_SS that are not None 
clusters_SS = [x for x in clusters_SS if x is not None]
print(len(clusters_SS))

junc_ind_keep = junction_ids_conversion[junction_ids_conversion["Cluster"].isin(clusters_SS)]["junction_id_index"]
final_data = final_data[final_data.junction_id_index.isin(junc_ind_keep)]
final_data.head()

In [ ]:
# update junction_ids_conversion file to only include junctions in clusters_SS 
# update junction_id_index now that we have removed junctions
junction_ids_conversion = junction_ids_conversion[junction_ids_conversion["Cluster"].isin(clusters_SS)]
junction_ids_conversion["new_junction_id_index"] = np.arange(junction_ids_conversion.shape[0])

In [ ]:
# re-order the remaining junctions and subset the counts matrices
final_data = final_data.merge(junction_ids_conversion, on = "junction_id_index")
final_data.sort_values(by = ["new_junction_id_index"], inplace = True)

to_keep = final_data["junction_id_index"].unique()   
junc_counts_sub = junc_counts.tocsr()[:,to_keep].tocoo()
cluster_counts_sub = cluster_counts.tocsr()[:,to_keep].tocoo()

In [ ]:
final_data.head()

NameError: name 'final_data' is not defined

In [ ]:
# some sanity checks to make sure correct counts are outputted given new indices 
print(junc_counts_sub.toarray()[392, 12785])
print(cluster_counts_sub.toarray()[392, 12785])

### Simulate Data

In [ ]:
simulated_counts, cell_types, cell_type_psi = simulate_junc_counts(cluster_counts_sub, cell_types=None, simulate_cell_types = True)

In [ ]:
# turn cell_type_psi into a dataframe wtih columns cell_state1 and cell_state_2 based on index of column 
cell_type_psi_df = pd.DataFrame(cell_type_psi.numpy())
cell_type_psi_df.columns = ["cell_state_" + str(col) for col in cell_type_psi_df.columns]
cell_type_psi_df["new_junction_id_index"] = np.arange(cell_type_psi_df.shape[0])
cell_type_psi_df["difference"] = cell_type_psi_df["cell_state_1"] - cell_type_psi_df["cell_state_0"]
# plot histogram of difference
sns.histplot(cell_type_psi_df["difference"])
# add a title called "Difference in simulated junction p(success) between cell states (K=2)"
plt.title("Difference in simulated junction p(success) between cell states (K=2)")

In [ ]:
from scipy.sparse import coo_matrix
true_psi = (simulated_counts / cluster_counts_sub)
sim_juncs_counts = simulated_counts
# how many in true_psi are not nan ~ 10% of the data 
np.sum(~np.isnan(true_psi)) / (np.sum(~np.isnan(true_psi)) + np.sum(np.isnan(true_psi)))

In [ ]:
#get cell indices for new_cell_type==0 and new_cell_type==1
cell_type_0 = np.where(cell_types == 0)[0]
cell_type_1 = np.where(cell_types == 1)[0]

In [ ]:
from scipy.special import expit, logit

true_psi_df = pd.DataFrame(true_psi)
true_psi_df = expit(true_psi_df)

In [ ]:
# take the difference using the full simulated PSI matrix since these are the actual values 
# get the difference between cell_state_1 and cell_state_0 for each junction using true_psi matrix 

# based on these differences assign positive and negative labels for junctions
# the cell_type_psi matrix is just a matrix of priors on the junctions 
# the true observed PSIs will depend on how many reads were in cluster in the cell  

In [ ]:
# label anything with absolute difference of 0.2 or less as not cell state associated 
cell_type_psi_df["true_label"] = np.where(abs(cell_type_psi_df["difference"]) >= 0.3, "positive", "negative")
cell_type_psi_df.sort_values(by = ["difference"], inplace = True)
cell_type_psi_df.tail()

In [ ]:
cell_type_psi_df.sort_values(by = ["new_junction_id_index"], inplace = True)
cell_type_psi_df.head()

In [ ]:
# plot observed PSI values for the two groups of cell states for each junction
def plot_sim_junc_psi(junc_index):
    cell1 = true_psi_df.iloc[cell_type_1, junc]
    cell0 = true_psi_df.iloc[cell_type_0, junc]
    # combine cell1 and cell0 into one dataframe and make violin plot 
    cell1_df = pd.DataFrame(cell1)
    cell1_df["cell_state"] = "cell_state_1"
    cell0_df = pd.DataFrame(cell0)
    cell0_df["cell_state"] = "cell_state_0"
    cell_df = pd.concat([cell1_df, cell0_df])
    cell_df.columns = ["PSI", "cell_state"]
    cell_df["junction_id"] = junc
    # keep only non Nan values 
    cell_df = cell_df[~cell_df["PSI"].isnull()]
    # get junc diff from simulated p(success) 
    junc_diff = cell_type_psi_df[cell_type_psi_df["new_junction_id_index"] == junc]["difference"].values[0]
    print("junction difference: " + str(junc_diff))
    # count how many in each cell state
    cell1 = cell_df[cell_df["cell_state"] == "cell_state_1"]
    cell0 = cell_df[cell_df["cell_state"] == "cell_state_0"]
    print("cell_state_1: " + str(len(cell1)))
    print("cell_state_0: " + str(len(cell0)))
    # plot violin plot
    # make plot less wide and tall
    plt.figure(figsize=(6, 4))
    sns.violinplot(x = "cell_state", y = "PSI",data = cell_df)
    # make title junction id 
    plt.title("simulated PSI across cells for junc = " + str(junc))
    # make range of y axis 0 to 1
    plt.ylim(0, 1)
    plt.show()

In [ ]:
plot_sim_junc_psi(12513)

In [ ]:
# sample 10 random junctions and plot them
juncs = np.random.choice(np.arange(true_psi_df.shape[0]), 10, replace = False)
for junc in juncs:
    plot_sim_junc_psi(junc)

In [ ]:
# make dataframe using the following columsn 
sim_junc_counts_flat = pd.DataFrame({"cell_id_index": sim_juncs_counts.row, "new_junction_id_index": sim_juncs_counts.col, "new_junc_count": sim_juncs_counts.data})
sim_junc_counts_flat.head()

# also add new cell type column 
sim_junc_counts_flat["new_cell_type"] = np.array(cell_types[sim_junc_counts_flat["cell_id_index"]])
sim_junc_counts_flat.head()

In [ ]:
# update junction counts in final_data object to be the simulated counts 
final_data = final_data.merge(sim_junc_counts_flat, on = ["cell_id_index", "new_junction_id_index"])
final_data.head()

### Prep simulated data as input into mixture model

In [ ]:
sim_data = final_data.copy() 
# drop the old junction counts and junction id index
sim_data.drop(columns = ["junc_count", "junction_id_index"], inplace = True)
# rename columns new_junction_id_index and new_junc_count to junction_id_index and junc_count
sim_data.rename(columns = {"new_junction_id_index": "junction_id_index", "new_junc_count": "junc_count"}, inplace = True)
sim_data.head()

In [ ]:
cell_index_tensor, junc_index_tensor, my_data = make_torch_data(sim_data, **float_type)

In [ ]:
len(sim_data.junction_id_index.unique())

### Check how well the data is simulated, is there cell type specific splicing?

In [ ]:
# let's visualize junction usage ratios for each cell type 


### Run mixture model

In [ ]:
K = 5
print(K)

In [ ]:
# make new cell_ids_conversion dataframe using new cell type column
cell_ids_conversion_new = sim_data[["cell_id_index", "cell_type", "cell_id", "new_cell_type"]].drop_duplicates()
cell_ids_conversion_new.head()

In [ ]:
# set random seed
torch.manual_seed(0)

num_trials = 10 # should also be an argument that gets fed in
num_iters = 100 # should also be an argument that gets fed in

# loop over the number of trials (for now just testing using one trial but in general need to evaluate how performance is affected by number of trials)
#reload(betabinomo_mix_singlecells)

start_time = time.time()

# Running for just one K and assessing similarity across trials 

results = [ calculate_CAVI(K, my_data, float_type, hypers, init_labels = None, num_iterations = num_iters) 
           for t in range(num_trials) ]


# write the above line use fstring
print(f"This took {time.time() - start_time} seconds")

In [ ]:
best = np.argmax([ g[-1][-1] for g in results ]) # final ELBO
print(f"The trial with the highest ELBO was {best}")
ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[best]
elbos_all = np.array(elbos_all)
plt.plot(elbos_all[1:]); plt.show()

In [ ]:
juncs_probs = ALPHA_f / (ALPHA_f+PI_f)   
 
plt.hist(juncs_probs.cpu().numpy().flatten(), 20)
plt.title('Histogram of learned junction probabilities') 
plt.xlabel('Probability of junction success')
plt.show()

In [ ]:
PHI_f_plot = pd.DataFrame(PHI_f.cpu().numpy())
PHI_f_plot['cell_id'] = cell_ids_conversion["cell_type"].to_numpy()
PHI_f_plot

In [ ]:
cell_ids_conversion_new.sort_values(by = ["cell_id_index"], inplace = True)
cell_ids_conversion_new.head()

In [ ]:
# Obtain cell type labels for every cell in the matrix also 
unique_cell_types = cell_ids_conversion_new['new_cell_type'].unique()
num_unique_types = len(unique_cell_types)
colors = sns.color_palette('Set1', n_colors=num_unique_types)  # You can use any color palette
cell_type_colors = {cell_type: color for cell_type, color in zip(unique_cell_types, colors)}
cell_types = cell_ids_conversion_new.new_cell_type.values
# Convert cell types to corresponding colors for rows and columns
row_colors = [cell_type_colors[cell_type] for cell_type in cell_types]
col_colors = [cell_type_colors[cell_type] for cell_type in cell_types]

In [ ]:
all_iters_results = check_cell_pairs(results, row_colors, col_colors, cell_type_colors, num_cells_to_plot=100)

In [ ]:
juncs_probs_df = pd.DataFrame(juncs_probs, columns = range(K))
# add "cell_state" to each column name 
juncs_probs_df.columns = ["cell_state_" + str(col) for col in juncs_probs_df.columns]
juncs_probs_df["new_junction_id_index"] = junction_ids_conversion.new_junction_id_index.values
# convert to juncs_probs to pandas dataframe and calculate mean and std across cell states/topics
juncs_probs_df["junction_id"] = junction_ids_conversion.junction_id.values

In [ ]:
sim_data

In [ ]:
def plot_juncObsUsage(junc_index):

    # print junction ID using junction_ids_conversion
    junc_id = junction_ids_conversion[junction_ids_conversion["new_junction_id_index"] == junc_index].junction_id.values[0]
    print(junc_id)

    # get data for just junc_index 
    junc_dat = sim_data[sim_data.junction_id == junc_id]
    print(junc_dat.new_cell_type.value_counts())
    junc_dat["juncratio"] = junc_dat.junc_count / junc_dat.cluster_count
    print(junc_dat["juncratio"])
    # make violin plot for junc_dat junction usage ratio coloured by cell_type and rotate plot 90 degrees
    plot = ggplot(junc_dat, aes(x='new_cell_type', y='juncratio', fill="new_cell_type")) + geom_violin() + geom_point() + plotnine.labels.ggtitle(junc_id) + plotnine.coords.coord_flip() 

    # add number of cells in each cell_type to plot 
    print(plot)

def plot_juncProbs(junc_index):
    
    # print junction ID using junction_ids_conversion
    print(junction_ids_conversion[junction_ids_conversion["new_junction_id_index"] == junc_index])
    junc_id = junction_ids_conversion[junction_ids_conversion["new_junction_id_index"] == junc_index].junction_id.values[0]
    
    # get data for just junc_index 
    junc_dat = juncs_probs_df[juncs_probs_df.new_junction_id_index == junc_index]
    junc_dat = junc_dat.melt().iloc[0:K]
    junc_dat.value = junc_dat.value.astype(float)
    # make violin plot for junc_dat junction usage ratio coloured by cell_type
    # don't print x-axis tick labels 
    plot = ggplot(junc_dat, aes(x='variable', y='value')) + geom_point() + theme(axis_text_x=element_blank())
    print(plot)

In [ ]:
scores_all_juncs = []
for junc_index in range(juncs_probs.shape[0]):
    a = ALPHA_f[junc_index, ]
    b = PI_f[junc_index, ]
    scores_all_juncs.append(score(a, b).item())

# turn scores_all_juncs into dataframe and add junction_id_index as a column
scores_all_juncs_df = pd.DataFrame(scores_all_juncs, columns = ["score"])
scores_all_juncs_df["new_junction_id_index"] = junction_ids_conversion.new_junction_id_index.values
scores_all_juncs_df.sort_values(by="score", ascending=False).head(10)
juncs_test = scores_all_juncs_df.sort_values(by="score", ascending=False).head(2).new_junction_id_index.values

In [ ]:
sim_data["new_cell_type"] = sim_data["new_cell_type"].astype(str)

In [ ]:
plot_juncObsUsage(4067)

In [ ]:
# for each junction in top10juncs_state1, run plot_juncObsUsage and plot_juncProbs
for junc in juncs_test:
    plot_juncObsUsage(junc)
    plot_juncProbs(junc)

In [ ]:
# convert PHI_f to a dataframe and add a column with cell ID and cell type 
PHI_f = pd.DataFrame(PHI_f)
# Add "CellState" to each column 
PHI_f.columns = ["CellState_" + str(i) for i in range(PHI_f.shape[1])]
PHI_f['cell_id'] = cell_ids_conversion_new.cell_id.values
PHI_f['cell_type'] = cell_ids_conversion_new.new_cell_type.values
PHI_f.groupby('cell_type').sum()

In [ ]:
PHI_f

In [ ]:
# group by cell_type and sum across each cellstate 
PHI_f.groupby('cell_type').sum()
sum_prop=PHI_f.groupby('cell_type').sum()/PHI_f.groupby('cell_type').count()
# remove cell_id column 
sum_prop=sum_prop.drop(columns=['cell_id'])
#masked_data = np.ma.masked_equal(sum_prop, 0)
sns.set(font_scale=0.8)  # Adjust font size for labels
# make figure bigger 
plt.figure(figsize=(10, 8))
# make font size of xtickts and yticks bigger
plt.yticks(fontsize=14)
plt.xticks(fontsize=14)
sns.heatmap(sum_prop, annot=True, fmt=".2f", cmap='viridis')

In [ ]:
# evaluate how well junctions are assigned to positive labels (associated with cell state) or negative (no association with cell state)
scores_all_juncs_df.sort_values(by="new_junction_id_index", inplace = True)
scores_all_juncs_df.head()

In [ ]:
# add true_label column to scores_all_juncs_df
scores_all_juncs_df["true_label"] = cell_type_psi_df["true_label"].values
scores_all_juncs_df.head()

In [ ]:
scores_all_juncs_df.shape

In [ ]:
from scipy.special import logit, expit
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc

In [ ]:
# make new column where label positive is 1 and negative is 0 in scores_all_juncs_df
scores_all_juncs_df["true_label_num"] = np.where(scores_all_juncs_df["true_label"] == "positive", 1, 0)
scores_all_juncs_df.head()

In [ ]:
scores_all_juncs_df["score"].describe()

In [ ]:
pre, rec, thres = precision_recall_curve(scores_all_juncs_df["true_label_num"], scores_all_juncs_df["score"])

# Calculate AUC-ROC
auc_roc = roc_auc_score(scores_all_juncs_df["true_label_num"], scores_all_juncs_df["score"])
print("The AUC ROC is: " + str(auc_roc))

auc_pr = auc(rec, pre)
print("the AUC_PR is: " + str(auc_pr))

In [ ]:
# plot precision recall curve using values calculated above
plt.figure(figsize=(10, 5))
plt.plot(rec, pre, color='black')
plt.xlabel('Recall')
plt.ylabel('Precision')